# 05 Decoder Based Response Generator



## Objective

Build a lightweight Transformer Decoder model for Controlled Natural Language Generation.

Input:
- Linearized intent + slots

Example:

[BOS] intent : order_status <sep> order_id : ORD123 [EOS]


Output:

Your order ORD123 is currently being processed.


## Concepts

- Token Embedding
- Positional Encoding
- Transformer Decoder
- Auto-regressive generation
- Cross Entropy Training

In [2]:
#Imports 
import json
import math
import time

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from tqdm import tqdm

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Device Setup
if torch.backends.mps.is_available():
    device = torch.device("mps")

elif torch.cuda.is_available():
    device = torch.device("cuda")

else:
    device = torch.device("cpu")


print("Using device:", device)

In [ ]:
# Load Vocabulary
DATA_PATH = "../data/"


with open(DATA_PATH + "token_to_id.json") as f:
    token_to_id = json.load(f)


with open(DATA_PATH + "id_to_token.json") as f:
    id_to_token = json.load(f)


# JSON keys become string
id_to_token = {
    int(k):v 
    for k,v in id_to_token.items()
}


VOCAB_SIZE = len(token_to_id)


print("Vocabulary Size:", VOCAB_SIZE)

In [ ]:
# Special Tokens

PAD_ID = token_to_id["[PAD]"]

BOS_ID = token_to_id["[BOS]"]

EOS_ID = token_to_id["[EOS]"]


print(
    "PAD:",
    PAD_ID,
    "BOS:",
    BOS_ID,
    "EOS:",
    EOS_ID
)

In [ ]:
# Tokenizer Functions
# concept : Text -> Numbers
# concept : Numbers -> Text
# 
def encode(text):

    tokens = text.split()

    ids = []

    for token in tokens:

        ids.append(
            token_to_id.get(
                token,
                token_to_id["[UNK]"]
            )
        )

    return ids



def decode(ids):

    words=[]

    for i in ids:

        if i in [
            PAD_ID,
            BOS_ID,
            EOS_ID
        ]:
            continue

        words.append(
            id_to_token.get(
                int(i),
                "[UNK]"
            )
        )


    return " ".join(words)

IndentationError: unexpected indent (441725316.py, line 4)

In [ ]:
#Test

sample = "[BOS] intent : order_status [EOS]"

ids = encode(sample)

print(ids)

print(decode(ids))

In [ ]:
# Load Training Data
# 
with open(
    DATA_PATH + "linearized_train.json"
) as f:

    train_data=json.load(f)


with open(
    DATA_PATH + "linearized_validation.json"
) as f:

    val_data=json.load(f)


print(
    len(train_data),
    len(val_data)
)

In [ ]:
# DataSet Class -> This converts JSON into tensors

MAX_LEN = 80


class NLGDataset(Dataset):


    def __init__(self,data):

        self.data=data



    def __len__(self):

        return len(self.data)



    def pad(self,ids):

        ids = ids[:MAX_LEN]

        ids += (
            [PAD_ID]
            *
            (MAX_LEN-len(ids))
        )

        return ids



    def __getitem__(self,index):

        item=self.data[index]


        source = encode(
            item["linearized_sequence"]
        )


        target = encode(
            "[BOS] "
            +
            item["target_text"]
            +
            " [EOS]"
        )


        return (

            torch.tensor(
                self.pad(source)
            ),


            torch.tensor(
                self.pad(target)
            )
        )

In [ ]:
# Data Loader

train_loader = DataLoader(

    NLGDataset(train_data),

    batch_size=16,

    shuffle=True
)


len(train_loader)

In [ ]:
# Positional Encoding

class PositionalEncoding(nn.Module):


    def __init__(self,d_model,max_len=100):

        super().__init__()


        pe=torch.zeros(
            max_len,
            d_model
        )


        position=torch.arange(
            0,max_len
        ).unsqueeze(1)


        div=torch.exp(

            torch.arange(
                0,
                d_model,
                2
            )

            *

            (-math.log(10000.0)/d_model)
        )


        pe[:,0::2]=torch.sin(position*div)

        pe[:,1::2]=torch.cos(position*div)


        self.pe=pe.unsqueeze(0)



    def forward(self,x):

        return x + self.pe[:,:x.size(1)].to(device)

In [ ]:
# Transformer Decoder Model -> Main Model   

class DecoderNLG(nn.Module):


    def __init__(self):

        super().__init__()


        d_model=128


        self.embedding = nn.Embedding(

            VOCAB_SIZE,

            d_model
        )


        self.position = PositionalEncoding(
            d_model
        )


        decoder_layer = nn.TransformerEncoderLayer(

            d_model=d_model,

            nhead=4,

            batch_first=True
        )


        self.transformer = nn.TransformerEncoder(

            decoder_layer,

            num_layers=2
        )


        self.output = nn.Linear(

            d_model,

            VOCAB_SIZE
        )



    def forward(self,x):


        x=self.embedding(x)


        x=self.position(x)


        x=self.transformer(x)


        return self.output(x)

In [ ]:
# Create Model

model = DecoderNLG().to(device)


loss_fn = nn.CrossEntropyLoss(

    ignore_index=PAD_ID
)


optimizer=torch.optim.Adam(

    model.parameters(),

    lr=0.001
)

In [ ]:
# Training

EPOCHS=20


for epoch in range(EPOCHS):

    model.train()

    total_loss=0


    for src,tgt in tqdm(train_loader):


        src=src.to(device)

        tgt=tgt.to(device)


        output=model(src)


        loss=loss_fn(

            output.reshape(
                -1,
                VOCAB_SIZE
            ),


            tgt.reshape(-1)

        )


        optimizer.zero_grad()


        loss.backward()


        optimizer.step()


        total_loss += loss.item()



    print(

        "Epoch",

        epoch+1,

        "Loss",

        total_loss

    )

In [ ]:
# Save Model

torch.save(

    model.state_dict(),

    "../models/decoder_model.pt"
)


print("Model saved")

In [ ]:
# Generate Response

def generate(text):

    model.eval()


    ids=encode(text)


    ids=torch.tensor(
        ids
    ).unsqueeze(0).to(device)


    with torch.no_grad():

        output=model(ids)


    prediction=torch.argmax(
        output,
        dim=-1
    )


    return decode(
        prediction[0].cpu().tolist()
    )

In [ ]:
# Test

sample=train_data[0]

print("INPUT:")
print(sample["linearized_sequence"])

print()

print("GENERATED:")
print(
    generate(
        sample["linearized_sequence"]
    )
)

print()

print("EXPECTED:")
print(
    sample["target_text"]
)

In [ ]:
# Save Predictions

predictions=[]


for item in val_data:


    predictions.append(

        {
            "input":
            item["linearized_sequence"],


            "expected":
            item["target_text"],


            "generated":
            generate(
                item["linearized_sequence"]
            )
        }

    )



with open(
    "../outputs/decoder_predictions.json",
    "w"
) as f:

    json.dump(
        predictions,
        f,
        indent=4
    )


print("Saved decoder predictions")